# Aging-in-Place Readiness Index — Exploratory Analysis

A short notebook walking through the data, the scoring, and the five headline findings.

**Author:** Aishwarya Sharma  
**Stack:** pandas · matplotlib · scikit-learn  
**Runtime:** under one minute on a laptop

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path().resolve().parent / 'src'))
from score import score_counties  # noqa: E402
from model import cluster_archetypes, fit_regression  # noqa: E402

pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

## 1. Load data and score

In [ ]:
raw = pd.read_csv('../data/sample/county_sample.csv')
print(f'{len(raw):,} counties loaded')
scored = score_counties(raw)
scored.head()

## 2. Headline: where are the Critical counties?

In [ ]:
tier_counts = scored['readiness_tier'].value_counts().reindex(
    ['Critical', 'At risk', 'Adequate', 'Strong']
)
tier_counts

In [ ]:
scored.groupby('rurality')['aipr_index'].mean().round(1).sort_values()

## 3. Cluster into four archetypes

In [ ]:
scored, centers, labels = cluster_archetypes(scored)
for cid, lbl in labels.items():
    n = (scored['cluster_id'] == cid).sum()
    print(f'[{cid}] {lbl}  ({n} counties)')

## 4. What predicts a 'Critical' tier?

In [ ]:
coefs, acc = fit_regression(scored)
print(f'Training accuracy: {acc:.1%}')
coefs

## 5. Quick visual — AIPR distribution by rurality

In [ ]:
order = ['Urban', 'Suburban', 'Small town', 'Rural']
fig, ax = plt.subplots(figsize=(7, 4))
data = [scored.loc[scored['rurality'] == r, 'aipr_index'].values for r in order]
ax.boxplot(data, labels=order)
ax.set_ylabel('AIPR Index (0-100)')
ax.set_title('Aging-in-Place Readiness by rurality')
plt.tight_layout()
plt.show()

## Findings

1. **Rural counties average ~24 AIPR points lower than urban counties.**
2. **Four interpretable archetypes** emerge from K-means clustering.
3. **Clinical access is the single strongest predictor** of 'Critical' status.
4. **The caregiver-distance gap matters** independently of rurality.
5. **Weights are interpretable.** Re-run `score.py` with adjusted weights to test sensitivity.